<a href="https://colab.research.google.com/github/Elakkiya1802/AI-hypertension/blob/main/BERT_doc_summarization_and_q_and_a_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install transformers==4.44.0 torch sentencepiece PyPDF2 python-docx -q

In [1]:
import transformers
print("Version:", transformers.__version__)  # Must show 4.44.0

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Version: 4.44.0


In [2]:
from transformers import pipeline

summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
print("✅ Summarization loaded")

qa_pipeline = pipeline("question-answering", model="bert-large-uncased-whole-word-masking-finetuned-squad")
print("✅ QA loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Some weights of the model checkpoint at bert-large-uncased-whole-word-masking-finetuned-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ QA loaded


In [3]:
import os
import PyPDF2
import docx

def load_document(filepath):
    ext = os.path.splitext(filepath)[1].lower()
    if ext == '.txt':
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read()
    elif ext == '.pdf':
        with open(filepath, 'rb') as f:
            reader = PyPDF2.PdfReader(f)
            text = " ".join(page.extract_text() or "" for page in reader.pages)
    elif ext == '.docx':
        doc = docx.Document(filepath)
        text = " ".join(p.text for p in doc.paragraphs if p.text.strip())
    else:
        raise ValueError("Unsupported format. Use .txt, .pdf, or .docx")
    return " ".join(text.split())

print("✅ File reader ready")

✅ File reader ready


In [4]:
def summarize_document(text):
    words = text.split()
    # chunk if document is long
    if len(words) > 900:
        chunks = [" ".join(words[i:i+900]) for i in range(0, len(words), 900)]
        summaries = []
        for i, chunk in enumerate(chunks):
            print(f"  Summarizing chunk {i+1}/{len(chunks)}...")
            out = summarizer(chunk, max_length=120, min_length=30, do_sample=False)
            summaries.append(out[0]["summary_text"])
        combined = " ".join(summaries)
        if len(combined.split()) > 200:
            out = summarizer(combined, max_length=150, min_length=50, do_sample=False)
            return out[0]["summary_text"]
        return combined
    else:
        out = summarizer(text, max_length=150, min_length=30, do_sample=False)
        return out[0]["summary_text"]

print("✅ Summarizer ready")

✅ Summarizer ready


In [5]:
def summarize_document(text):
    words = text.split()
    # chunk if document is long
    if len(words) > 900:
        chunks = [" ".join(words[i:i+900]) for i in range(0, len(words), 900)]
        summaries = []
        for i, chunk in enumerate(chunks):
            print(f"  Summarizing chunk {i+1}/{len(chunks)}...")
            out = summarizer(chunk, max_length=120, min_length=30, do_sample=False)
            summaries.append(out[0]["summary_text"])
        combined = " ".join(summaries)
        if len(combined.split()) > 200:
            out = summarizer(combined, max_length=150, min_length=50, do_sample=False)
            return out[0]["summary_text"]
        return combined
    else:
        out = summarizer(text, max_length=150, min_length=30, do_sample=False)
        return out[0]["summary_text"]

print("✅ Summarizer ready")

✅ Summarizer ready


In [6]:
def answer_question(context, question):
    words = context.split()
    if len(words) > 400:
        chunks = [" ".join(words[i:i+400]) for i in range(0, len(words), 400)]
        best_answer, best_score = "Not found.", -1
        for chunk in chunks:
            result = qa_pipeline(question=question, context=chunk)
            if result["score"] > best_score:
                best_score = result["score"]
                best_answer = result["answer"]
        return best_answer
    else:
        return qa_pipeline(question=question, context=context)["answer"]

print("✅ QA ready")

✅ QA ready


In [8]:
# CHANGE THIS to your uploaded file name
FILE_PATH = "/content/Sample FAQ Document.docx"

document = load_document(FILE_PATH)
print(f"✅ Loaded — {len(document.split())} words\n")

print("=" * 50)
print("SUMMARY")
print("=" * 50)
summary = summarize_document(document)
print(summary)

✅ Loaded — 337 words

SUMMARY
SmartShop has a secure payment system. All payments are processed through encrypted and secure payment gateways. Refunds are processed within 5–7 business days after successful cancellation or return.


In [10]:
question = input("Ask a question: ")
answer = answer_question(document, question)
print("\nAnswer:", answer)

Ask a question: what is the email ID for customer support

Answer: support@smartshop.com
